In [14]:
import numpy as np 
import pandas as pd
from pandas import Series,DataFrame
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
#表一取E-W
data1=pd.read_excel('data/表1-患者列表及临床信息.xlsx')
#表二取C-X
data2=pd.read_excel('data/表2-患者影像信息血肿及水肿的体积及位置.xlsx')
data2=data2.drop(['ID', '首次检查流水号','随访1流水号','随访2流水号','随访3流水号','随访4流水号','随访5流水号','随访6流水号','随访7流水号','随访8流水号'],axis=1)

#对表一中性别与血压数据进行处理,男为1，女为0,血压拆分为血压比，血高压，血低压
def convert(x):
        if x=='男':
            return 1
        else:
            return 0
data1['性别']= data1['性别'].map(convert)
def convert(item):
        high,low = item.split("/")
        ratio=eval(high)/eval(low)
        return ratio,eval(high),eval(low)
result = data1['血压'].apply(convert)
data1['血压'] = [x[0] for x in result]
data1['血高压'] = [x[1] for x in result]
data1['血低压'] = [x[2] for x in result]
#合并表一表二
data= pd.concat([data1, data2], axis=1)
data=data.iloc[:,4:]
#去除data中流水号，拆分训练组与测试组
x_train=data[:100]
x_test=data[100:]
y_train=data1['90天mRS'][:100] 
y_test=data1['90天mRS'][100:]
np.set_printoptions(suppress=True)
conde=np.isnan(data)
data[conde]=0
#x_train,x_cv,y_train,y_cv=train_test_split(x_train,y_train,test_size=0.2)
display(x_train.shape,y_train.shape,x_test.shape,y_test.shape)

(100, 219)

(100,)

(60, 219)

(60,)

In [15]:
#随机森林算法
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
rf = RandomForestClassifier(n_estimators=200,n_jobs=-1)
# x_train = np.ascontiguousarray(x_train)
# x_test = np.ascontiguousarray(x_test)
scaler=StandardScaler()
x_train_s=scaler.fit_transform(x_train)
x_test_s=scaler.fit_transform(x_test)
data_s=scaler.fit_transform(data)
rf.fit(x_train_s,y_train)
y_=rf.predict(data_s)
print("rf模型的估计准确率：",rf.score(x_train_s,y_train))
#输出所有患者血肿扩张预测概率（右边为扩张概率）,你自己导出一下



rf模型的估计准确率： 1.0


In [16]:
result_df = pd.DataFrame()


result_df['患者ID'] = data1['Unnamed: 0']
result_df['90天mRS'] = pd.DataFrame(y_)[0]

result_df.to_excel('./result_3b.xlsx')

In [17]:
#逻辑回归算法
from sklearn.linear_model  import LogisticRegression
lf = LogisticRegression()
lf.fit(x_train,y_train)
y_=lf.predict(data)
print("lf模型的估计准确率：",lf.score(x_train,y_train))
y_

lf模型的估计准确率： 0.51


array([1., 3., 3., 1., 3., 5., 2., 5., 3., 5., 0., 0., 2., 5., 2., 5., 3.,
       0., 5., 0., 0., 5., 2., 0., 5., 3., 6., 3., 5., 5., 5., 0., 3., 4.,
       5., 3., 3., 1., 1., 1., 2., 0., 3., 4., 3., 5., 5., 3., 2., 5., 5.,
       3., 2., 3., 5., 0., 1., 5., 2., 4., 4., 2., 2., 1., 3., 6., 2., 2.,
       6., 5., 3., 5., 2., 0., 3., 5., 6., 2., 3., 1., 3., 2., 5., 5., 2.,
       2., 6., 2., 2., 0., 3., 5., 6., 2., 4., 3., 0., 5., 3., 2., 6., 6.,
       6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 0.,
       6., 6., 6., 0., 6., 6., 6., 6., 6., 6., 6., 3., 6., 3., 2., 6., 3.,
       5., 6., 5., 4., 2., 3., 5., 6., 0., 1., 5., 0., 2., 0., 3., 6., 2.,
       5., 2., 5., 6., 3., 3., 3.])

In [18]:
#Knn算法
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
#对输入数据进行归一化（Z-Score规范化）
scaler=StandardScaler()
x_train_s=scaler.fit_transform(x_train)
x_test_s=scaler.fit_transform(x_test)
data_s=scaler.fit_transform(data)
#建立模型
knn=KNeighborsClassifier(n_neighbors=3,n_jobs=-1)
#跑模型
knn.fit(x_train_s,y_train)
#输出预测结果左边为0概率，右边为1概率
y_=knn.predict(data_s)
#利用训练模型大致估计准确度
print("knn模型的估计准确率：",knn.score(x_train_s,y_train))
y_

knn模型的估计准确率： 0.55


array([3., 0., 5., 4., 2., 4., 0., 4., 3., 1., 1., 1., 1., 2., 2., 5., 3.,
       0., 1., 0., 3., 2., 0., 1., 1., 1., 1., 3., 1., 1., 4., 0., 1., 3.,
       0., 3., 3., 2., 0., 1., 3., 0., 5., 1., 3., 0., 2., 1., 1., 1., 0.,
       3., 2., 3., 3., 0., 1., 5., 2., 0., 0., 0., 2., 3., 0., 0., 2., 2.,
       4., 2., 0., 2., 1., 0., 3., 5., 6., 1., 1., 1., 3., 0., 5., 1., 2.,
       0., 1., 2., 2., 0., 1., 1., 0., 5., 4., 2., 2., 6., 2., 2., 2., 1.,
       1., 2., 0., 2., 2., 2., 2., 3., 2., 3., 0., 2., 1., 1., 3., 2., 1.,
       2., 2., 1., 2., 2., 1., 2., 1., 2., 2., 1., 4., 2., 5., 5., 2., 1.,
       2., 0., 3., 1., 5., 4., 3., 3., 2., 0., 3., 1., 1., 2., 5., 1., 4.,
       3., 1., 3., 4., 3., 1., 2.])

In [19]:
import tensorflow as tf

# 构建模型
model = tf.keras.Sequential([
    tf.keras.layers.Reshape((219, 1), input_shape=(219,)),
    tf.keras.layers.Conv1D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Conv1D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Conv1D(64, 3, activation='relu'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

# 编译模型
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 转换数据形状
x_train=np.array(x_train)
x_train = x_train.reshape(100, 219)

# 训练模型
model.fit(x_train, y_train, epochs=10, batch_size=32)


Epoch 1/10
4/4 [==============================] - 1s 7ms/step - loss: 1113.9861 - accuracy: 0.1700
Epoch 2/10
4/4 [==============================] - 0s 8ms/step - loss: 606.6794 - accuracy: 0.1400
Epoch 3/10
4/4 [==============================] - 0s 9ms/step - loss: 348.2687 - accuracy: 0.1900
Epoch 4/10
4/4 [==============================] - 0s 7ms/step - loss: 218.1826 - accuracy: 0.1900
Epoch 5/10
4/4 [==============================] - 0s 7ms/step - loss: 127.3657 - accuracy: 0.2200
Epoch 6/10
4/4 [==============================] - 0s 7ms/step - loss: 48.9426 - accuracy: 0.2000
Epoch 7/10
4/4 [==============================] - 0s 7ms/step - loss: 14.1013 - accuracy: 0.2100
Epoch 8/10
4/4 [==============================] - 0s 7ms/step - loss: 4.5437 - accuracy: 0.2500
Epoch 9/10
4/4 [==============================] - 0s 7ms/step - loss: 3.5890 - accuracy: 0.2400
Epoch 10/10
4/4 [==============================] - 0s 7ms/step - loss: 2.7312 - accuracy: 0.2400


In [20]:
x_test=np.array(x_test)
y_=model.predict(x_train)
y_1=np.argmax(y_, axis=1)
display(y_1,np.array(y_train))

4/4 [==============================] - 0s 3ms/step


array([3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 2, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], dtype=int64)

array([4., 0., 5., 4., 3., 5., 2., 4., 3., 3., 1., 0., 1., 2., 2., 5., 3.,
       0., 1., 2., 3., 2., 1., 1., 2., 3., 6., 4., 4., 5., 4., 0., 5., 4.,
       5., 3., 3., 2., 1., 1., 3., 0., 5., 3., 1., 2., 1., 3., 1., 1., 0.,
       3., 2., 3., 4., 0., 3., 5., 0., 4., 4., 2., 2., 3., 3., 6., 2., 2.,
       6., 3., 1., 5., 1., 0., 3., 5., 6., 1., 1., 1., 3., 2., 5., 1., 2.,
       0., 1., 2., 1., 5., 2., 5., 2., 5., 4., 4., 2., 5., 3., 2.])